In [1]:
# Setup
import os
os.chdir('/home/smallyan/eval_agent')

# Load environment variables
import subprocess
result = subprocess.run(['bash', '-c', 'source /home/smallyan/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        if key in ['HF_HOME', 'HF_TOKEN', 'OPENAI_API_KEY', 'NDIF_API_KEY', 'HUGGINGFACE_HUB_CACHE']:
            os.environ[key] = value

print(f"Working directory: {os.getcwd()}")
print(f"HF_HOME: {os.environ.get('HF_HOME', 'NOT SET')}")

Working directory: /home/smallyan/eval_agent
HF_HOME: /net/projects2/chai-lab/shared_models


# Generalizability Evaluation for Belief Tracking Repository

## Overview
This notebook evaluates the generalizability of findings in the belief-tracking research repository.

Repository path: `/net/scratch2/smallyan/belief_tracking_eval`

## Research Summary:
The original work investigates how language models (Llama-3-70B-Instruct, Llama-3.1-405B-Instruct) track character beliefs using a "lookback mechanism". Key findings include:
- Answer payload localizes at layers 56+ at final token
- Answer pointer at layers 34-52 at final token  
- Binding address and payload at layers 33-38 at state token
- Binding source reference (character/object) at layers 20-34

## Evaluation Checklist:
- **GT1**: Generalization to a New Model
- **GT2**: Generalization to New Data  
- **GT3**: Method / Specificity Generalizability

In [2]:
import sys
sys.path.insert(0, '/net/scratch2/smallyan/belief_tracking_eval')
sys.path.insert(0, '/net/scratch2/smallyan/belief_tracking_eval/src')

import json
import random
import torch
from src.dataset import Dataset, Sample

# Check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load entity data
data_dir = '/net/scratch2/smallyan/belief_tracking_eval/data'
with open(f'{data_dir}/synthetic_entities/characters.json', 'r') as f:
    all_characters = list(json.load(f))
with open(f'{data_dir}/synthetic_entities/bottles.json', 'r') as f:
    all_objects = list(json.load(f))
with open(f'{data_dir}/synthetic_entities/drinks.json', 'r') as f:
    all_states = list(json.load(f))

print(f"\nLoaded {len(all_characters)} characters, {len(all_objects)} objects, {len(all_states)} states")

CUDA available: True
GPU: NVIDIA A40
Memory: 47.7 GB

Loaded 103 characters, 21 objects, 23 states


---
## GT1: Generalization to a New Model

**Objective**: Test if the belief tracking mechanism findings generalize to a model NOT used in the original work.

**New Model**: Mistral-7B-Instruct-v0.3
- Different model family (Mistral vs Llama)
- Not used in original research
- 32 layers (vs 80 layers in Llama-70B)

**Test Approach**:
1. Run causal mediation analysis with interchange interventions
2. Check if information localization patterns scale proportionally to model depth
3. Verify lookback mechanism exists

In [3]:
# Load Mistral-7B-Instruct-v0.3
from nnsight import LanguageModel

hf_cache = '/net/projects2/chai-lab/shared_models/hub'

print("Loading Mistral-7B-Instruct-v0.3...")
model = LanguageModel(
    'mistralai/Mistral-7B-Instruct-v0.3',
    device_map='cuda',
    dtype=torch.float16,
    dispatch=True,
    cache_dir=hf_cache,
)
print("Model loaded!")
print(f"Number of layers: {len(model.model.layers)}")

Loading Mistral-7B-Instruct-v0.3...


tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

In [4]:
# Continue - check model loaded
print(f"Model type: {type(model)}")
print(f"Layers: {len(model.model.layers)}")

In [5]:
# Check if model was loaded
print("Checking model...")
print(model)

In [6]:
# Debug - basic test
a = 1
print(f"a = {a}")